# Кредитный скоринг физических лиц
Учебный проект LendingClub. **Ограничение:** `Charged Off` — прокси списания, не доказательство просрочки 90+ дней. Статусы завершённых кредитов могут приводить к смещению выборки. Все расчёты прибыли условные. Не используйте для реальных кредитных решений.

Ноутбуки запускаются последовательно 01 → 02 → 03 → 04. Каталог `artifacts/` сохраняет промежуточные результаты. Один раз разместите архив `accepted_2007_to_2018Q4.csv.gz` в текущем каталоге Colab или укажите путь в ячейке ниже. При новом сеансе Colab локальные файлы пропадают: используйте Google Drive или загрузите готовые промежуточные файлы.

Зависимости: `pip install pandas numpy matplotlib seaborn scikit-learn lightgbm optbinning shap joblib pyyaml`.

## 01. EDA и временная выборка

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
os.makedirs('artifacts', exist_ok=True)

# Отредактируйте при необходимости; файл НЕ включаем в GitHub.
DATA_FILE='accepted_2007_to_2018Q4.csv.gz'
assert os.path.exists(DATA_FILE), f'Файл не найден: {DATA_FILE}'
cols=['issue_d','loan_status','loan_amnt','installment','annual_inc','dti','delinq_2yrs','fico_range_low','fico_range_high','open_acc','pub_rec','revol_bal','total_acc','revol_util']
df=pd.read_csv(DATA_FILE,compression='gzip',usecols=lambda c:c in cols,low_memory=False)
print('Размер исходных данных:',df.shape)
display(pd.DataFrame({'type':df.dtypes.astype(str),'missing_pct':(100*df.isna().mean()).round(2)}))
print(df['loan_status'].value_counts(dropna=False))


In [ ]:
df['issue_d']=pd.to_datetime(df['issue_d'],format='%b-%Y',errors='coerce')
df=df[df['loan_status'].isin(['Fully Paid','Charged Off']) & df['issue_d'].notna()].copy()
df['target']=(df['loan_status']=='Charged Off').astype(int)
df=df.drop(columns=['loan_status']).sort_values('issue_d').reset_index(drop=True)
# 70/15/15 приблизительно по строкам, но строго по границам месяцев.
month_counts=df.groupby('issue_d').size().sort_index().cumsum()
n=len(df)
m_train=(month_counts - int(n*.70)).abs().idxmin()
m_valid=(month_counts - int(n*.85)).abs().idxmin()
assert m_train<m_valid, 'Нужны разные месяцы границ'
train=df[df.issue_d<=m_train].copy()
valid=df[(df.issue_d>m_train)&(df.issue_d<=m_valid)].copy()
test=df[df.issue_d>m_valid].copy()
assert train.issue_d.max()<valid.issue_d.min()<test.issue_d.min()
for name,part in [('train',train),('valid',valid),('test',test)]:
    part.to_pickle(f'artifacts/{name}.pkl')
    print(name,len(part),f'{len(part)/n:.1%}',part.issue_d.min().date(),part.issue_d.max().date())
print('Доля списаний train:',train.target.mean())


**Контроль:** весь список использованных признаков ограничен набором из исходных заявочных данных. Точная доступность `installment` на момент решения и соответствие дефолту 90+ должны быть согласованы с преподавателем. Test не применяется к выбору признаков и параметров.